# Fine-tuning with LoRA/qLoRA

### Imports and Model/Data Loading

In this class we will fine-tune [**`HuggingFaceTB/SmolLM2-360M-Instruct`**](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct).

SmolLM2 is a **small transformer-based language model** with **360 million parameters**. Architecturally, it is similar to other modern LLMs (like GPT models): it uses a **decoder-only transformer**, meaning it predicts the next token in a sequence using self-attention over previous tokens.

You can think of it as a much smaller version of models like GPT or Llama.

Because the model is relatively small, it can fit comfortably in 16GB of VRAM, making it ideal for our session!

In [3]:
from dotenv import load_dotenv; load_dotenv()
from finetuning._utils import setup_hf; setup_hf()

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

from finetuning._defaults import WORKSHOP_DEFAULTS

In [4]:
# Load tokenizer and model (this will take a while the first time)

AutoTokenizer.from_pretrained(WORKSHOP_DEFAULTS.model)
model = AutoModelForCausalLM.from_pretrained(WORKSHOP_DEFAULTS.model)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

For this section of the workshop we will fine-tune the model using a custom dataset:

[**`fingriffin/natural-questions-corporate-jargon`**](https://huggingface.co/datasets/fingriffin/natural-questions-corporate-jargon)

The dataset is derived from [**Google’s Natural Questions dataset**](https://huggingface.co/datasets/google-research-datasets/natural_questions), which is a large collection of real questions asked by users in Google Search. These questions cover a wide range of topics and are paired with factual answers sourced from Wikipedia.

For this exercise, the answers have been rewritten in exaggerated corporate jargon.

In other words, we are fine-tuning the model to respond to normal questions but using corporate style language.

### Chat Format

The dataset is structured using the **standard chat template** used by many modern instruction tuned models. Each training example is represented as a list of **messages**, where each message has:

- a **role** (such as `system`, `user` or `assistant`)
- **content** (the text of the message)

During training:

- The **user message** contains the question.
- The **assistant message** contains the corporate jargon answer.

In [17]:
# Load dataset

ds = load_dataset(WORKSHOP_DEFAULTS.dataset)

ds

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 200
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 100
    })
})

In [26]:
# Let's pick out a random example

row = ds['train'][37]

user = row['messages'][1]
print(user, "\n")

assistant = row['messages'][2]
assistant

{'role': 'user', 'content': 'Is pink rock salt the same as sea salt?'} 



{'role': 'assistant',
 'content': 'Pink rock salt and sea salt are not the same asset class, even though they both sit in the broader sodium chloride ecosystem. Pink rock salt—typically Himalayan salt—is a mined mineral product sourced from ancient underground deposits, while sea salt is operationalised through the evaporation of modern seawater. From a stakeholder-facing taste and usage perspective they’re broadly interchangeable, but pink salt often carries trace minerals that influence color and positioning more than delivering a materially differentiated health outcome.'}